# 06 · Write-up
**How to read it:** section 1 is the answer; sections 2–3 are the background you need; section 4 is a
notebook-by-notebook tour; sections 6 onwards are the write-up proper.

### What this notebook answers from the brief
| Brief asks for | Where |
|---|---|
| Your shortlist of surviving precursors | sections 1 and 6 (each rule written out in full) |
| The candidates you rejected and why | section 7 |
| A brief description of your assumptions | section 9 (incl. how the rules are tested: built on dev, tested on holdout, no leakage) |
| The data source, with enough detail to reproduce | section 10 |
| Your main design choices (H, W, thresholds, how you defined normal) | section 11 (incl. why H = 12, W = 36, with precedent) |
| If you run out of time: what you'd do next | section 12 |
| If you used AI tools: transcripts and notes on corrections | section 13 and the README |
| Background: build a small version of Oden's root-cause workflow | section 5 |
| (also) Why the simple rules beat an ML model | section 8 |

In [1]:
# --- Setup ----------------------------------------------------------------------------
# Libraries, plus the project's helper files in ../src (data.py, detectors.py, evaluation.py).
# Every analysis function this notebook calls lives there, in numbered sections (§1 ... §9).
import json, sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="IProgress not found")  # harmless: kagglehub's progress bar

import pandas as pd
import plotly.io as pio

# Two outputs per chart: Plotly's own MIME type (rendered natively by VS Code and JupyterLab, works
# offline) plus an HTML fallback for classic Jupyter (loads plotly.js from a CDN).
pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.max_colwidth", 120, "display.width", 200)  # show long rule texts in full

ROOT = Path.cwd().parent            # project root (this notebook lives in notebooks/)
sys.path.insert(0, str(ROOT / "src"))  # so `from data import ...` finds ../src/data.py

In [2]:
# --- The saved results of notebooks 01-05 ------------------------------------------------
# This notebook computes nothing: it reads what notebooks 01-05 saved to outputs/. Run them first, in order.
from evaluation import load_results, check_results   # src/evaluation.py §9
display(check_results()[["ran_at", "status"]])        # when each notebook ran, and whether it is still up to date

N1 = load_results("01_dataset_and_parameters")        # dataset profile, timestamp audit, look-alikes
N2 = load_results("02_candidates")                    # the ten candidates
N3 = load_results("03_ml_benchmark")                  # the ML models' scores
N4 = load_results("04_test_and_select")               # every candidate's scores, gates and verdict; the H x W grid
N5 = load_results("05_audit")                         # the audits
K = N4["kinds"]["any"]                                # "any failure", the main event type

,ran_at,status
notebook,,
01_dataset_and_parameters,2026-11-02 04:06:47,up to date
02_candidates,2026-11-02 04:06:50,up to date
03_ml_benchmark,2026-11-02 04:08:07,up to date
04_test_and_select,2026-11-02 04:17:07,up to date
05_audit,2026-11-02 04:17:22,up to date


## 1 · The answer in one paragraph
On the Azure Predictive Maintenance data (719 failure events, hourly readings, 100 machines, all of 2015),
each part's own sensor signal shifts about **48 hours** before it fails. But each part's signal also shifts
~470 times a year *without* a failure, so a sensor alone can't be trusted (at most ~37% of its alarms are real).
What separates real from false shifts is **the part's own error code**, logged 24 hours before the failure.
Alerts are scored on a window of **H = 12, W = 36**: an alert counts if it comes between 48 and 12 hours before a
failure, so every alert that counts leaves at least half a day to act. The strongest precursor, **C09 "matched
signature"** (a part's own signal drifting *and* its own error code logged), scores **F1 0.94 on machines it never
saw**: it catches **99%** of failures, **90%** of its alarms are real, and it warns about **24 hours** ahead. Two
more survive: **C10** (drift on a worn part) warns earlier (~43 h) but is right only 41% of the time, and **C08**
(any drift + any error) is a simpler fallback. The data is synthetic and unusually clean, so treat the absolute
numbers as optimistic and the method as the transferable result.

## 2 · Key terms
| Term | Meaning |
|---|---|
| **Event** | a component failure. Failures on the same machine at the same hour count as one event (the brief's rule). "Any failure" is the main target; each part (comp1–comp4) is also run on its own. |
| **Precursor / rule** | a pattern that tends to come before a failure, written as a simple yes/no rule checked every hour ("alert" or "no alert"). |
| **H and W** | for a failure at time T, an alert counts if it falls between T − (H + W) and T − H. Here **H = 12, W = 36**: an alert counts if it comes between 48 and 12 hours before the failure. |
| **Normal operation** | hours far from any failure: not in the 48 h before one, not in the 24 h after a failure or repair, and not in the first 24 h of data. |
| **Lift** | how much more often a rule fires before failures than in normal operation (2 = twice as often). |
| **Precision** | of the alerts a rule raises, the share followed by a failure in time. |
| **Coverage** | of the failures, the share that had at least one alert in their window. |
| **F1** | the single score used to pick rules: `2 × precision × coverage / (precision + coverage)`. It is high only when *both* are high, so it can't be gamed by never firing or always firing. |
| **Dev / holdout machines** | the 100 machines are split 50/50. The rules are **built and tuned on the dev machines** and **tested on the holdout machines**, which play no part in building them. |
| **Look-alike drift** | a signal shift that looks like a pre-failure shift but isn't followed by a failure (~470 per part per year). |

### Why F1, and not lift or coverage alone
Each of the brief's three numbers can be made to look good by a rule that is useless in practice:
* **Lift alone** rewards a rule that almost never fires: one lucky alert just before a failure gives an enormous
  lift while missing nearly every failure.
* **Coverage alone** rewards a rule that is always on: it "catches" every failure, but almost all of its alarms
  are false, and the crew soon ignores it.
* **Precision alone** rewards firing only on the most obvious cases, again missing most failures.

What an operator needs is **both**: most failures caught (coverage) *and* most alarms real (precision). F1 combines
the two as their harmonic mean, which stays low unless both are high. For example, when the ideas were compared,
"any signal drifts" (C05) caught 98% of failures but only 28% of its alarms were real, so its F1 is just 0.44, even
though a plain average of the two would flatter it at 0.63. The matched signature (C09) is high on both (90% of its
alarms real, 99% of failures caught) and scores 0.94. Lift is
still checked (each rule must reach at least 2) and reported, but as a sanity check rather than the target.

### What a "gate" is
With 655 versions of the ten ideas tried, some will score well by luck or by quietly using information from
the future. So before ranking, every idea must pass **four gates** on the dev machines:
1. **Not luck:** it must clearly beat copies of itself shifted randomly in time (which keep how often it fires but
   break any link to failures), with a stricter bar because ten ideas are tested.
2. **Lift ≥ 2:** it fires at least twice as often before failures as in normal operation.
3. **Coverage ≥ 5%:** it catches a meaningful share of failures.
4. **Causal check:** set up using only January–June, it must keep at least half its F1 on July–December.

Ideas that pass are ranked by dev F1 and the top three kept (skipping any that is the same alarm as a better
one). Nine of the ten passed every gate; only C07 (overdue maintenance) failed, on the causal check.

## 3 · How the project fits together, and where the numbers come from
**Two kinds of files:**
* **`src/` (three helper files)** hold every function: `data.py` (settings, loading, windows), `detectors.py`
  (the rules and the ten candidates), `evaluation.py` (metrics, the selection protocol, audits, ML benchmark,
  and saving results).
* **Notebooks 01–06.** Run them **in order**. Each of 01–05 computes its own analysis live from the raw data
  and, in its last cell, saves its key results to `outputs/`:

| Run | Computes | Saves |
|---|---|---|
| 01 Dataset and parameters | the dataset profile, the timestamp audit, look-alike drifts, confidence | `outputs/01_dataset_and_parameters.json` |
| 02 Candidates | the ten candidates and their variants | `outputs/02_candidates.json` |
| 03 ML benchmark (~2 min) | three ML models, trained and scored | `outputs/03_ml_benchmark.json` |
| 04 Test and select (~10 min) | the full selection protocol for all five event types, and the H × W grid | `outputs/04_test_and_select.json` |
| 05 Audit | the audits of the detectors 04 kept (it reads 04's file for which ones) | `outputs/05_audit.json` |
| 06 Write-up (this one) | nothing: it reads the five files above | — |

**Why save results at all?** So this write-up can summarise everything without re-running ~15 minutes of
analysis, and so every number quoted here comes from the notebook that produced it. Each file records the
settings it was made with (H, W, seed) and the time it ran.

**Guarding against out-of-date files.** The table at the top of this notebook lists each file with the time
it ran. It says "re-run" if the code in `src/` changed afterwards, or if notebook 04 was re-run after notebook
05 (05 audits 04's detectors), and it warns if the notebooks used different H and W settings.

## 4 · What each notebook found

Each subsection gives the notebook's key conclusions, then the saved numbers behind them.

### 4.1 · Notebook 01: dataset and parameters
| Characteristic | Finding |
|---|---|
| Events | 761 failure rows → **719 events** (same machine + hour merged); 131–259 per part |
| Sampling and gaps | hourly, 100 machines × 8,761 h (all of 2015); no gaps, no missing values |
| Clustering | only 3 events come within a week of the previous one on the same machine; failures cluster *by machine* (0–19 each) |
| Operating regimes | effectively one: the four machine models run at the same signal levels, and machines never stop |
| Counters vs. live readings | the four sensors (volt, rotate, pressure, vibration) are live readings; the only counter is "hours since maintenance", which we build from the maintenance log |
| Timestamps | failures are logged in a daily 06:00 batch, but an hour-by-hour check finds no post-failure data before the stamp |
| Where the warning signs are | each part has one signature signal (comp1 → volt, comp2 → rotate, comp3 → pressure, comp4 → vibration). It shifts ~48 h before failure; the part's own error code lands exactly 24 h before |
| Key property | ~470 look-alike drifts per part never end in a failure, capping a signal-only rule at 21–34% precision; the error code is in 96–100% of real drifts but only 14–18% of look-alikes |
| Confidence | 363 holdout events for "any failure" (±5 points), 65 for comp3 (±12); the data is synthetic, so treat absolute numbers as optimistic |

In [3]:
# Saved by notebook 01: the dataset in numbers, and the look-alike drifts behind the key property.
e, g = N1["events"], N1["gaps"]
print(f"{e['failure_rows']} failure rows -> {e['events']} events ({e['multi_part']} with 2+ parts); per part: {e['per_part']}")
print(f"{N1['sampling']['machines']} machines x {N1['sampling']['hours']:,} hourly readings, {N1['sampling']['start'][:10]} to {N1['sampling']['end'][:10]}")
print(f"median {g['median_days_between_events']:.0f} days between events on a machine; {g['events_within_7_days']} within a week")
pd.DataFrame(N1["lookalikes"]).set_index("index")

761 failure rows -> 719 events (42 with 2+ parts); per part: {'comp1': 192, 'comp2': 259, 'comp3': 131, 'comp4': 179}
100 machines x 8,761 hourly readings, 2015-01-01 to 2016-01-01
median 30 days between events on a machine; 3 within a week


,signal,episodes_ending_in_failure,episodes_without_failure,precision_ceiling,share_with_error_failure,share_with_error_none,precision_ceiling_with_error
index,,,,,,,
comp1,volt,172,464,0.270,0.959,0.144,0.711
comp2,rotate,248,475,0.343,0.992,0.183,0.739
comp3,pressure,126,483,0.207,1.000,0.153,0.630
comp4,vibration,171,476,0.264,1.000,0.153,0.701


### 4.2 · Notebook 02: candidate precursors
Ten candidates in four families, each with a physical story and a yes/no detector (table below).

Ideas came from domain knowledge (bearing wear, friction, blockage, electrical faults), from what notebook 01
found in the data, and from the ML models in notebook 03. Each idea is built in many variants (655 in total),
so ideas are compared fairly rather than on a lucky first draft.

In [4]:
# Saved by notebook 02: the ten candidates.
pd.DataFrame(N2["candidates"])[["id", "title", "family", "variants"]]

,id,title,family,variants
0,C01,Voltage creeps up,telemetry,95
1,C02,Rotation sags,telemetry,95
2,C03,Pressure builds,telemetry,95
3,C04,Vibration rises,telemetry,95
4,C05,Any signal drifts,combo,95
5,C06,Any error code,errors,20
6,C07,Overdue maintenance,counter,20
7,C08,Drift confirmed by an error,combo,40
8,C09,Matched signature (part's own signal + own error),combo,40
9,C10,Drift on an aging part,combo,60


### 4.3 · Notebook 03: ML benchmark
* Three models (gradient boosting with and without the maintenance counters, and logistic regression), trained
  on dev machines from 32 features and scored exactly like the rules. The best, logistic regression, reaches
  F1 ~0.63, well below C09; section 8 compares them and explains why the rules win.
* **What the models lean on:** smoothed signals (6 h and 24 h averages) and each part's own error counts, which
  pointed the candidates toward "smoothed drift + error code".

In [5]:
# Saved by notebook 03: each ML model on the holdout machines.
pd.DataFrame(N3["models"])[["detector", "F1", "F1_CI", "precision", "coverage"]].round(3)

,detector,F1,F1_CI,precision,coverage
0,ML: logistic regression,0.629,"[0.59, 0.66]",0.464,0.974
1,"ML: gradient boosting, no maintenance counters",0.512,"[0.48, 0.54]",0.344,1.000
2,"ML: gradient boosting, all features",0.301,"[0.26, 0.34]",0.177,1.000


### 4.4 · Notebook 04: test and select
* **All ten ideas** were scored with lift, precision and coverage, with thresholds from normal operation only.
  Nine passed every gate; **C07 failed the causal check**. Why the others lose: section 7.
* **Kept:** **C09**, **C08** and **C10** (section 6 has their scores and the rules in full).
* **Smoothing:** a 6 h average works best; CUSUM is worst.
* **Choosing H and W:** one F1 grid of 100 windows. F1 is flat for every window that contains the error code
  logged 24 h before a failure, and collapses once the window starts 36 h or more ahead. The window is
  **H = 12, W = 36** (section 11).
* **Every event type:** C09 ranks first for any failure and for each part separately.

In [6]:
# Saved by notebook 04: the shortlist for every event type, and the H x W grid's key numbers.
display(pd.DataFrame([dict(event=e, rank=k["kept"].index(c["cid"]) + 1, id=c["cid"], idea=c["title"],
                           F1=c["hold"]["f1"], precision=c["hold"]["precision"], coverage=c["hold"]["coverage"])
                      for e, k in N4["kinds"].items() for c in k["concepts"] if c["status"] == "kept"]
                     ).sort_values(["event", "rank"]).round(3))
G = pd.DataFrame(N4["hw_grid"]).set_index("H")
G.columns = G.columns.astype(int)
print(f"H x W grid (dev F1): chosen H=12, W=36 -> {G.loc[12, 36]:.3f} | best anywhere -> {G.values.max():.3f} | "
      f"windows starting >= 36 h ahead: at most {G.loc[[36, 48, 72]].values.max():.2f}")

,event,rank,id,idea,F1,precision,coverage
1,any,1,C09,Matched signature (part's own signal + own error),0.941,0.898,0.989
0,any,2,C08,Drift confirmed by an error,0.744,0.628,0.914
2,any,3,C10,Drift on an aging part,0.564,0.413,0.889
4,comp1,1,C09,Matched signature (part's own signal + own error),0.926,0.891,0.965
5,comp1,2,C10,Drift on an aging part,0.566,0.406,0.930
3,comp1,3,C01,Voltage creeps up,0.412,0.266,0.919
8,comp2,1,C09,Matched signature (part's own signal + own error),0.943,0.899,0.992
7,comp2,2,C08,Drift confirmed by an error,0.778,0.694,0.886
6,comp2,3,C06,Any error code,0.234,0.622,0.144
10,comp3,1,C09,Matched signature (part's own signal + own error),0.930,0.869,1.000


H x W grid (dev F1): chosen H=12, W=36 -> 0.936 | best anywhere -> 0.948 | windows starting >= 36 h ahead: at most 0.24


### 4.5 · Notebook 05: audit (trying to break the results)
Notebook 05 tries to break the shortlist with the classic traps. Each passes:
* **Luck from trying many ideas.** Notebook 05 builds 300 fake rules from pure random noise (no link to failures
  at all), tunes them exactly like the real rules, and keeps the one that scores best on the dev machines. Even
  that "winner" scores only about **0.08** on dev and on holdout, far below C09's 0.94. So picking the best
  of many tries manufactures almost no skill, and the kept rules also beat shuffled-in-time copies of themselves
  by a wide margin.
* **Leakage through the failure hour.** Scoring the failure hour itself would turn "a repair was logged" into a
  fake perfect rule; the failure hour is never scored.
* **Counters that leak the future.** "Hours since maintenance" looks excellent until it may only use past data,
  then collapses; the causal gate rejects it (C07).
* **Time drift.** Rules set up on January–June alone still work on July–December (table below).
* **Different machines.** C09 works on every machine model and age band (precision 0.87–0.93, coverage 0.97–0.99).
* **Clustered failures.** Too rare to matter (3 within a week of each other), and the intervals resample whole
  machines anyway.

In [7]:
# Saved by notebook 05: the luck check and the time-drift check, in plain terms.
L, T = N5["audits"]["lucky"], {t["cid"]: t for t in N5["audits"]["temporal"]}
pd.DataFrame([
    dict(check=f"Best of {len(L['dev_f1'])} rules built from random noise, picked on dev machines",
         result=f"F1 {L['best_dev']:.2f} on dev, {L['best_hold']:.2f} on holdout",
         meaning="trying many rules finds almost nothing by luck"),
    *[dict(check=f"{cid}: set up on Jan–Jun only, scored on Jul–Dec",
           result=f"F1 {T[cid]['first_half_thresholds']['f1']:.2f} (vs {T[cid]['full_year_thresholds']['f1']:.2f} set up on the full year)",
           meaning="still works on months it has not seen")
      for cid in ("C09", "C08", "C10")],
]).set_index("check")

,result,meaning
check,,
"Best of 300 rules built from random noise, picked on dev machines","F1 0.08 on dev, 0.08 on holdout",trying many rules finds almost nothing by luck
"C09: set up on Jan–Jun only, scored on Jul–Dec",F1 0.93 (vs 0.93 set up on the full year),still works on months it has not seen
"C08: set up on Jan–Jun only, scored on Jul–Dec",F1 0.74 (vs 0.73 set up on the full year),still works on months it has not seen
"C10: set up on Jan–Jun only, scored on Jul–Dec",F1 0.57 (vs 0.55 set up on the full year),still works on months it has not seen


## 5 · How this mirrors Oden's Agentic Root Cause Analysis workflow
The brief asks for "a small version" of the workflow in Oden's post,
[Agentic Root Cause Analysis](https://engineering.oden.io/blog/agentic-root-cause-analysis). Each of its steps
has a counterpart here:

| Oden's workflow | This project |
|---|---|
| **1. Build operating context** (line diagrams, process knowledge, operator interviews) | Notebook 01: EDA of the five raw files, the dataset profile, and an audit of how failures are timestamped |
| **2. Reduce the search space with ML** (gradient-boosted classifiers, feature importances) | A gradient-boosting / logistic-regression benchmark and its feature importances, used for ideas, not as a detector (**notebook 03**) |
| **3. Generate hypotheses** that tie a signal pattern to a physical mechanism and a testable statistic | Notebook 02: 10 candidates, each with a story and a fixed yes/no detector |
| **4. Filter by lift analysis**: lift, precision, and permutation tests against random nominal windows | Notebook 04: lift, precision and coverage for every candidate; a circular-shift null test (a permutation test that keeps each detector's firing pattern); thresholds from nominal ("normal") running only, with downtime-like periods masked |
| **5. Root-cause families, playbooks, alerts** | The shortlist forms one family per part: C09 (own signal + own error code) is the "act now" alarm, and C10 (drift on an aging part) the earlier watchlist. Playbooks would come next, with plant input |

Two points from the post that this data echoes:
* **Lift and precision understate real causes**, partly because operators sometimes prevent the event. Here, ~470
  look-alike excursions per part never end in a failure (notebook 01, section 7). Some may be failures that were
  averted. That caps the precision of any signal-only rule and is why the error-code confirmation matters.
* **Atomic, explainable detectors are preferred.** C09 is one pattern applied to each part. Per component it is a
  single two-condition rule (own signal drifting AND own error code logged), as the brief allows: "a candidate can
  use one signal or combine several".

## 6 · Shortlist: the surviving precursors ("any failure", holdout machines, H = 12, W = 36)

In [8]:
# Saved by notebook 04: the three kept rules on holdout machines (window H=12, W=36).
pd.DataFrame([dict(id=c["cid"], precursor=c["title"], F1=c["hold"]["f1"], precision=c["hold"]["precision"],
                   coverage=c["hold"]["coverage"], lift=c["hold"]["lift"], lead_h=c["lead_median"],
                   alerts_per_machine_month=c["hold"]["alerts_per_machine_month"])
              for c in K["concepts"] if c["status"] == "kept"]).round(3)

,id,precursor,F1,precision,coverage,lift,lead_h,alerts_per_machine_month
0,C08,Drift confirmed by an error,0.744,0.628,0.914,27.959,24.0,2.192
1,C09,Matched signature (part's own signal + own error),0.941,0.898,0.989,236.379,24.0,1.478
2,C10,Drift on an aging part,0.564,0.413,0.889,16.108,43.0,4.131


**Conclusion.** Three precursors survive. **C09** (a part's own signal drifting *and* its own error code) is
the alarm to act on: 90% precision, 99% coverage, ~1.5 alerts per machine per month, ~24 h ahead. **C10** (drift
on an aging part) is an earlier, noisier watchlist (~43 h ahead, 41% precision). **C08** (any drift + any error)
is the fallback that needs no part ↔ code pairing.

### The three rules, written out
Each rule is checked **every hour on every machine**, and alerts when **both** of its conditions hold.
* **6 h average:** the mean of a sensor's last 6 hourly readings.
* **Usual level and spread:** each machine's median reading and how much its readings normally vary
  (a robust standard deviation: 1.4826 × the median absolute deviation). A machine is compared with itself.
* **Each part's own sensor and error code(s):**

| Part | Sensor (direction of a fault) | Own error code(s) |
|---|---|---|
| comp1 | voltage (rises) | error1 |
| comp2 | rotation (falls) | error2 or error3 |
| comp3 | pressure (rises) | error4 |
| comp4 | vibration (rises) | error5 |

**C09 · Matched signature** (the alarm to act on). For any one part:
1. its sensor's 6 h average is **more than 3.09 spreads beyond the machine's usual level**, in the fault direction
   (on a typical machine: voltage above ~190 vs ~170 usually, rotation below ~383 vs ~449, pressure above ~113 vs
   ~100, vibration above ~47 vs ~40), **and**
2. that part's **own error code** was logged **at least once in the last 24 hours**.

**C08 · Drift + any error** (the fallback):
1. **any** of the four sensors' 6 h average is **more than 3.72 spreads beyond the machine's usual level**, in its
   fault direction, **and**
2. **any** error code was logged **at least once in the last 48 hours**.

**C10 · Drift on an aging part** (the early warning). For any one part:
1. its sensor's 6 h average is **more than 3.72 spreads beyond the machine's usual level**, in the fault
   direction, **and**
2. the part has gone **longer since its last replacement than usual**: about 33 days for comp1, 31 for comp2,
   34 for comp3 and 33 for comp4.

## 7 · Rejected candidates, and why

In [9]:
# Every idea that was not kept, with the protocol's reason (which gate it failed, or what outranked it).
pd.DataFrame([dict(id=c["cid"], idea=c["title"], hold_F1=c["hold"]["f1"], reason=c["reason"])
              for c in K["concepts"] if c["status"] != "kept"]).round(3)

,id,idea,hold_F1,reason
0,C01,Voltage creeps up,0.281,"passed all gates but outranked by three stronger, non-redundant detectors"
1,C02,Rotation sags,0.387,"passed all gates but outranked by three stronger, non-redundant detectors"
2,C03,Pressure builds,0.240,"passed all gates but outranked by three stronger, non-redundant detectors"
3,C04,Vibration rises,0.289,"passed all gates but outranked by three stronger, non-redundant detectors"
4,C05,Any signal drifts,0.440,"passed all gates but outranked by three stronger, non-redundant detectors"
5,C06,Any error code,0.546,"passed all gates but outranked by three stronger, non-redundant detectors"
6,C07,Overdue maintenance,0.000,collapses with past-only thresholds (Jul–Dec F1 0.00 vs 0.05 with full-year thresholds): its skill leaks from the fu...


**Conclusion.** Six of the seven rejected ideas are real but weaker signals that were outranked: single signals
(C01–C04) are fooled by look-alike drifts, "any signal drifts" (C05) raises too many false alarms, and "any error
code" (C06) misses half the failures. Only **C07 (overdue maintenance)** failed a gate: its skill leaks from the
future (notebook 05, section 3).

## 8 · Why the simple rules win
Notebook 03 trained three ML models on every feature the rules could use (32 per machine-hour), then scored
them exactly like the rules: same window, same alert threshold method (P99 of normal operation), same holdout
machines. The table below puts them next to the three kept rules. (ML scores shift by about ±0.02 between
runs, because training uses a random sample of normal hours.)

In [10]:
# The three kept rules (notebook 04) next to the three ML models (notebook 03): same window, same holdout machines.
rules = [dict(detector=f"rule: {c['cid']} {c['title']}", F1=c["hold"]["f1"], precision=c["hold"]["precision"],
              coverage=c["hold"]["coverage"]) for c in K["concepts"] if c["status"] == "kept"]
ml = pd.DataFrame(N3["models"])[["detector", "F1", "precision", "coverage"]].to_dict("records")
pd.DataFrame(rules + ml).sort_values("F1", ascending=False).round(3)

,detector,F1,precision,coverage
1,rule: C09 Matched signature (part's own signal + own error),0.941,0.898,0.989
0,rule: C08 Drift confirmed by an error,0.744,0.628,0.914
3,ML: logistic regression,0.629,0.464,0.974
2,rule: C10 Drift on an aging part,0.564,0.413,0.889
4,"ML: gradient boosting, no maintenance counters",0.512,0.344,1.000
5,"ML: gradient boosting, all features",0.301,0.177,1.000


**Why a two-condition rule beats a 32-feature model here:**
* **The precursor has a sharp, specific structure:** *a part's own signal drifts while its own error code is
  logged*. C09 encodes that directly. A generic model trained on "any failure vs normal" has to discover the
  part-specific pairing from the data, and the linear model can't express it at all.
* **Look-alikes fool anything that scores signal levels.** Each part's signal drifts ~470 times a year without
  a failure (notebook 01, section 7). Only the matching error code separates real drifts from look-alikes, and
  the rule demands it explicitly.
* **The rules are tuned on the score we report** (alert-level F1). The models are trained to rank individual
  hours, then thresholded once, which is a weaker fit to the goal.
* **The models catch every failure but raise far more false alarms**, and gradient boosting with counters leans
  on "hours since maintenance", the same leaky signal the causal gate rejects (notebook 05, section 3).

**Limits:** no hyperparameter search, one alert threshold, and one "any failure" output rather than one per
part. This is a sanity ceiling, not the best possible ML. A rule that is also easier to explain and trust is the
better choice unless a model clearly beats it, and here none does.

## 9 · Assumptions
* **Failure timestamps mark the end of the degradation.** They are logged in a daily 06:00 batch, but an
  hour-by-hour check (notebook 01, section 3) finds no post-failure data before them.
* **The repair logged with each failure is part of the failure,** not a precursor.
* **Telemetry is a true hourly reading** at its timestamp, and every rule uses only past hours.
* **Machines run continuously** (minimum rotation 138 rpm), so there is no idle mode to exclude.
* **Machines are alike enough** that 50 dev machines can inform 50 holdout machines (split within each model).
* **The look-alike drifts** (signal shifts with no failure after them) are real behaviour of this data, perhaps
  averted failures or simulator noise. They are not relabelled.

### How the rules are tested (no leakage)
**Built on dev, tested on holdout.** Everything is decided on the 50 dev machines: which ideas, their settings and
cut-offs, which sensor and error codes belong to which part, the window, and the features the ML models lean on.
The 50 holdout machines are used only to score the finished rules, and their failure dates are used only to grade
the alerts. One subtlety: a sensor rule compares a machine with its *own* usual level. For a holdout machine that
level comes from its own readings over the year, never from its failure dates. In real use it would come from
past readings only; notebook 05, section 4 checks that rules set up from January–June still work on July–December.

**What is scored** (350 of the 363 holdout failures):
* **The failure hour itself is never scored:** it contains the repair record, which would be a fake "precursor".
* **The 24 h after a failure or repair are muted** (alerts there don't count). Right after a repair the crew is
  already at the machine, and the readings reflect the restart, not normal running. It hides almost nothing:
  only 1 of 719 failures came within 24 h of the previous one, and muting 0 to 72 h barely changes the results
  (notebook 05, section 4).
* **The first 24 h of 2015 are not scored:** the 6 h and 24 h averages and error counts need a day of history.
  The 13 holdout failures left out are all on 2 January 2015, whose window would start in 2014.
* **Alerts in the last 48 h of the year are not judged:** the data ends before we would know whether a failure
  followed. No failures are dropped; it only affects which alerts count toward precision.

## 10 · Data source and reproduction
* **Data:** [Microsoft Azure Predictive Maintenance on Kaggle](https://www.kaggle.com/datasets/arnabbiswas1/microsoft-azure-predictive-maintenance),
  version 3, five CSVs. It downloads automatically with
  `kagglehub.dataset_download("arnabbiswas1/microsoft-azure-predictive-maintenance")`; no account needed.
* **Code:** three helper files in `src/`. Run notebooks 01–06 in order; 01–05 each save their results to
  `outputs/` for this notebook (section 3). Machine split: seed 7, stratified by model.

## 11 · Main design choices
| Choice | Value | Why |
|---|---|---|
| Event | any component failure; same machine + same timestamp = one event (also run per component) | as the brief defines it; toggled in `Config(event=...)` |
| **H** (lead time) | **12 h** | every alert that counts leaves at least half a day to act; see "Why H = 12, W = 36" below |
| **W** (window) | **36 h** | reaches back to where each part's signal starts drifting, 48 h before a failure, so the window covers the whole precursor |
| **Normal operation** | not in the 48 h before any failure; not in the 24 h after a failure or repair; not in the first 24 h | the brief's definition, plus a recovery period and a warm-up for the averages |
| **Cut-offs** | sensors: a number of spreads beyond each machine's usual level; error codes: at least one; part age: longer than usual | set on the dev machines from normal operation, as the brief requires (section 9) |
| Selection | F1 of precision and coverage on dev; gates: null test, lift ≥ 2, coverage ≥ 5%, causal check | F1 can't be gamed by never firing or always firing; gates guard against luck and leakage |

### Why H = 12, W = 36
* **H = 12: half a day of warning.** With H = 0 an alert one hour before a failure would count as a success,
  which leaves no time to act. With H = 12, only alerts at least 12 hours ahead count.
* **W = 36: the whole precursor.** Each part's signal starts drifting about 48 hours before it fails, and its own
  error code is logged 24 hours before. A window from 48 to 12 hours before the failure contains both.
* **It costs nothing in detection.** On the F1 grid (notebook 04, section 6) the best window anywhere scores 0.948
  and H = 12, W = 36 scores 0.936, within the noise (~±0.02). A bigger buffer would not work: F1 collapses once the
  window starts 36 hours or more ahead.

**Why 12 hours? Precedent.** No standard prescribes a single number, but the principle and the unit are both
well established:
* **The warning must be longer than the time needed to act.** In reliability-centred maintenance this is the
  *net P-F interval*: the time between a fault becoming detectable (P) and the failure (F), minus the time needed
  to respond, must be "long enough to be of some use" ([Reliabilityweb, *RCM: On-Condition Task Interval
  Determination*](https://reliabilityweb.com/articles/entry/rcm-on-condition-task-interval-determination), after
  Moubray's RCM2). The international standard for machine prognostics, **ISO 13381-1** (*Condition monitoring and
  diagnostics of machines: Prognostics*, clause 5.3 "Setting alert, alarm and trip limits"), likewise bases alarm
  limits on the **maintenance lead time required**, including planning time and spare-part delivery
  ([ISO 13381-1:2004 preview](https://www.sis.se/api/document/preview/905442/)).
* **One shift is the natural unit of response.** Plants that run around the clock commonly work **12-hour
  shifts**, and longer shifts are favoured precisely because extended maintenance jobs can be finished within
  one crew's shift rather than handed over ([Power Engineering, *Making the shift: 12-hour vs. 8-hour*](https://www.power-eng.com/business/people/making-the-shift12-hour-vs-8-hour/)).
  A 12-hour lead means the crew on shift when the alert fires can diagnose the problem, get the part and do the
  repair, or hand a planned job to the next shift, before the failure.

The right number for a real line should come from its own repair times and part lead times; 12 hours is a
sensible default.

## 12 · What we would do next
* Ask whether the look-alike excursions precede maintenance that averted a failure. If so, some "false
  alarms" are really saves.
* Replace F1 with an expected-cost score once the cost of a false alarm and of a missed failure are known.
* Test how much lead time error codes really give with a non-batched error log. Here they land exactly 24 h
  before the failure (notebook 01, section 3).
* Train a per-part ML model (one output per component) and score it with the same alert-level metrics.
* Repeat the dev/holdout split over many seeds to measure how stable the selection is.